<a href="https://colab.research.google.com/github/RikyVisma/Finance-bros/blob/main/the%20relationship%20between%20internships%20dropout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Relevant Variables**
 **textcred_mat_practicas**: Number of internship credits in which the student enrolled.
**practicas**: Number of internship credits successfully passed.
**pract1**: Credits passed in internships. This may contain information similar to practicas.

In [ ]:
def classify_internship(row):
    enrolled = row["cred_mat_practicas"]
    passed = row["practicas"]

    if enrolled == 0:
        return "No internship"
    elif enrolled > 0 and passed == 0:
        return "Enrolled but not passed"
    elif passed < enrolled:
        return "Partially completed"
    else:
        return "Fully completed"

df["internship_status"] = df.apply(classify_internship, axis=1)

**2. Calculate the Internship Completion Rate**
Interpretation:

0 means that the student passed none of the enrolled internship credits.
0.5 means that the student completed 50% of the internship credits.
1 means that the student completed all internship credits.
NaN means that the student did not enroll in an internship.


In [ ]:
import numpy as np

df["internship_completion_rate"] = np.where(
    df["cred_mat_practicas"] > 0,
    df["practicas"] / df["cred_mat_practicas"],
    np.nan
)

**3. Compare Dropout Rates**

Assuming:

dropout = 1 means the student dropped out.
dropout = 0 means the student continued studying.

In [ ]:
internship_summary = (
    df.groupby("internship_status")["dropout"]
      .agg(
          total_students="count",
          dropout_students="sum",
          dropout_rate="mean"
      )
      .reset_index()
)

internship_summary["dropout_rate_percent"] = (
    internship_summary["dropout_rate"] * 100
)

print(internship_summary)

**4. Visualize the Relationship**

In [ ]:
import matplotlib.pyplot as plt

plot_data = (
    df.groupby("internship_status")["dropout"]
      .mean()
      .mul(100)
      .sort_values()
)

plot_data.plot(kind="bar")

plt.xlabel("Internship Status")
plt.ylabel("Dropout Rate (%)")
plt.title("Dropout Rate by Internship Status")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

**5. Test the Relationship Statistically**
A chi-square test can be used because both internship status and dropout are categorical variables.

Interpretation:

If p-value < 0.05, there is a statistically significant association between internship status and dropout.
If p-value >= 0.05, there is not enough evidence to conclude that internship status is associated with

In [ ]:
from scipy.stats import chi2_contingency

table = pd.crosstab(
    df["internship_status"],
    df["dropout"]
)

chi2, p_value, dof, expected = chi2_contingency(table)

print("Chi-square statistic:", chi2)
print("P-value:", p_value)